# Notebook 09 — GridSearchCV con Validación Cruzada Estratificada (K=5)

## Objetivo
Ejecutar **GridSearchCV** con **StratifiedKFold (K=5)** de forma **individual** para:
- **SVM** — `LinearSVC`
- **Regresión Logística** — `LogisticRegression`

Sobre el corpus completo de entrenamiento (`train_val_set.csv`), registrando para cada modelo:
- **F1-Score Macro** (mejor resultado del GridSearch)
- **Tiempo Total de Entrenamiento** (en minutos)
- **Pico Máximo de RAM** (medido con `memory_profiler`, en GB)

### Formato de salida esperado
```
SVM:           F1-Score: 0.XX | Tiempo: XX min | RAM: X GB
Reg. Logística: F1-Score: 0.XX | Tiempo: XX min | RAM: X GB
```

In [ ]:
import memory_profiler
print(f"memory_profiler versión: {memory_profiler.__version__}")

memory_profiler versión: 0.61.0


## Celda 1 — Importaciones

In [3]:
import pandas as pd
import time
import warnings
import gc

# Monitoreo de RAM
from memory_profiler import memory_usage

# scikit-learn
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# imbalanced-learn
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')
print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## Celda 2 — Carga del Corpus Completo

In [4]:
INPUT_PATH = 'data/train_val_set.csv'

df = pd.read_csv(INPUT_PATH, sep=';')
df['texto_lineal'] = df['texto_lineal'].fillna('')

X = df[['texto_lineal']]
y = df['label']

print(f"Corpus cargado.")
print(f"   Total de registros : {len(df):,}")
print(f"   Distribución de clases:\n{y.value_counts().to_string()}")

Corpus cargado.
   Total de registros : 45,784
   Distribución de clases:
label
1    26680
0    19104


## Celda 3 — Configuración del Pipeline Base y la Validación Cruzada

In [6]:
# Validación Cruzada Estratificada (K=5)
cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Submuestreo para balance de clases
undersampler = RandomUnderSampler(random_state=42)

# Vectorizador TF-IDF sobre la columna 'texto_lineal'
preprocessor = ColumnTransformer(
    transformers=[('tfidf', TfidfVectorizer(), 'texto_lineal')]
)

# Pipeline molde (el clasificador se reemplaza por cada malla)
pipeline_base = Pipeline(steps=[
    ('undersampler', undersampler),
    ('vectorizer', preprocessor),
    ('classifier', LogisticRegression())  # placeholder
])

print("Pipeline base y CV estratificado (K=5) configurados.")

Pipeline base y CV estratificado (K=5) configurados.


## Celda 4 — Helper: función de entrenamiento + medición de RAM

Se usa `memory_profiler.memory_usage()` con `interval=0.1` s para registrar el
**pico máximo de RAM (en MB)** durante toda la ejecución del GridSearchCV.
Después se convierte a GB para el reporte final.

In [7]:
def entrenar_con_metricas(grid_search_obj, X_data, y_data, nombre_modelo):
    """
    Ejecuta grid_search_obj.fit(X_data, y_data) midiendo:
      - Pico de RAM (memory_profiler) en GB
      - Tiempo total en minutos
    Retorna: (best_score, tiempo_min, ram_gb)
    """
    print(f"\n{'='*60}")
    print(f"  Entrenando: {nombre_modelo}")
    print(f"  (GridSearchCV K=5 — corpus completo)")
    print(f"{'='*60}")

    # Función interna que realiza el fit
    def _fit():
        grid_search_obj.fit(X_data, y_data)

    # Cronómetro
    t_inicio = time.time()

    # memory_usage() llama a _fit() en un proceso hijo y mide la RAM
    # interval: muestrea cada 0.1 s | include_children: incluye sub-procesos
    muestras_ram = memory_usage(
        (_fit, [], {}),
        interval=0.1,
        include_children=True,
        max_usage=False,
        retval=False
    )

    t_fin = time.time()

    # Cálculo de métricas
    pico_ram_mb = max(muestras_ram)
    pico_ram_gb = pico_ram_mb / 1024
    tiempo_min  = (t_fin - t_inicio) / 60
    best_score  = grid_search_obj.best_score_

    print(f"\n  Entrenamiento finalizado.")
    print(f"  Mejor configuración: {grid_search_obj.best_params_}")
    return best_score, tiempo_min, pico_ram_gb

## Celda 5 — GridSearchCV para SVM (LinearSVC)

In [8]:
from sklearn.base import clone

# Malla de hiperparámetros para SVM
param_grid_svm = [
    {
        'vectorizer__tfidf__max_features': [5000, 10000],
        'classifier': [LinearSVC(max_iter=2000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0]
    }
]

grid_svm = GridSearchCV(
    estimator=clone(pipeline_base),
    param_grid=param_grid_svm,
    cv=cv_estratificado,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

f1_svm, tiempo_svm, ram_svm = entrenar_con_metricas(
    grid_svm, X, y, "SVM — LinearSVC"
)

# Liberar memoria antes del siguiente modelo
gc.collect()
print("\nMemoria liberada. Listo para Regresión Logística.")


  Entrenando: SVM — LinearSVC
  (GridSearchCV K=5 — corpus completo)
Fitting 5 folds for each of 6 candidates, totalling 30 fits

  Entrenamiento finalizado.
  Mejor configuración: {'classifier': LinearSVC(max_iter=2000, random_state=42), 'classifier__C': 1.0, 'vectorizer__tfidf__max_features': 5000}

Memoria liberada. Listo para Regresión Logística.


## Celda 6 — GridSearchCV para Regresión Logística

In [10]:
# Malla de hiperparámetros para Regresión Logística
param_grid_log = [
    {
        'vectorizer__tfidf__max_features': [5000, 10000],
        'classifier': [LogisticRegression(max_iter=1000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0]
    }
]

grid_log = GridSearchCV(
    estimator=clone(pipeline_base),
    param_grid=param_grid_log,
    cv=cv_estratificado,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

f1_log, tiempo_log, ram_log = entrenar_con_metricas(
    grid_log, X, y, "Regresión Logística"
)

gc.collect()
print("\nMemoria liberada.")


  Entrenando: Regresión Logística
  (GridSearchCV K=5 — corpus completo)
Fitting 5 folds for each of 6 candidates, totalling 30 fits

  Entrenamiento finalizado.
  Mejor configuración: {'classifier': LogisticRegression(max_iter=1000, random_state=42), 'classifier__C': 10.0, 'vectorizer__tfidf__max_features': 5000}

Memoria liberada.


## Celda 7 — Tabla de Resultados Finales

In [11]:
print("\n" + "="*65)
print("  RESULTADOS FINALES — GridSearchCV K=5 (Corpus Completo)")
print("="*65)
print(f"  SVM:            F1-Score: {f1_svm:.2f} | Tiempo: {tiempo_svm:.1f} min | RAM: {ram_svm:.2f} GB")
print(f"  Reg. Logística: F1-Score: {f1_log:.2f} | Tiempo: {tiempo_log:.1f} min | RAM: {ram_log:.2f} GB")
print("="*65)

# Tabla resumen en DataFrame
resultados = pd.DataFrame({
    'Modelo':       ['SVM (LinearSVC)', 'Regresión Logística'],
    'F1-Score Macro': [round(f1_svm, 4), round(f1_log, 4)],
    'Tiempo (min)': [round(tiempo_svm, 2), round(tiempo_log, 2)],
    'RAM Pico (GB)': [round(ram_svm, 3), round(ram_log, 3)],
    'Mejor C':      [
        grid_svm.best_params_.get('classifier__C', 'N/A'),
        grid_log.best_params_.get('classifier__C', 'N/A')
    ],
    'max_features': [
        grid_svm.best_params_.get('vectorizer__tfidf__max_features', 'N/A'),
        grid_log.best_params_.get('vectorizer__tfidf__max_features', 'N/A')
    ]
})

display(resultados.set_index('Modelo'))


  RESULTADOS FINALES — GridSearchCV K=5 (Corpus Completo)
  SVM:            F1-Score: 0.90 | Tiempo: 0.5 min | RAM: 2.05 GB
  Reg. Logística: F1-Score: 0.90 | Tiempo: 0.4 min | RAM: 2.10 GB


,F1-Score Macro,Tiempo (min),RAM Pico (GB),Mejor C,max_features
Modelo,,,,,
SVM (LinearSVC),0.9035,0.48,2.052,1.0,5000
Regresión Logística,0.9041,0.44,2.100,10.0,5000
